In [ ]:
import tensorflow as tf

In [ ]:
tf.__version__

In [ ]:
import glob
import imageio
import matplotlib.pyplot as plt
import numpy as np
import os
import PIL
from tensorflow.keras import layers
import time

from IPython import display
import cv2

In [ ]:
#(train_images, train_labels), (_, _) = tf.keras.datasets.mnist.load_data()

In [ ]:

train_images = []
path = "/path/to/private-data/Datagen_Left/Inlet (5 marking)/train/*"
for directory_path in glob.glob(path):
    label=directory_path.split("\\")[-1]
    for image_path in glob.glob(os.path.join(directory_path,"*.bmp")):
        image = cv2.imread(image_path)

        # Convert to grayscale
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        #image = image/255


        if label == "Inlet (5 marking)":
            train_images.append(image)

print('Done')

            


In [ ]:
#path_train = "/path/to/private-data/Datagen_Left/Inlet (5 marking)/train"

In [ ]:
train_images

In [ ]:
import tensorflow as tf

def load_images_from_single_class(directory, class_name, img_height, img_width, batch_size):
    datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)
    class_directory = f"{directory}/{class_name}/"

    data_generator = datagen.flow_from_directory(
        directory=class_directory,  # Target directory
        target_size=(img_height, img_width),  # Resizing the images
        batch_size=batch_size,
        class_mode=None,  # Only data, no labels
        shuffle=True
    )
    return data_generator

# Example usage
#directory = "/path/to/private-data/Datagen_Left/Inlet (5 marking)"
#class_name = 'train'
#train_images = load_images_from_single_class(directory, class_name, img_height=400, img_width=320, batch_size=32)


In [ ]:
train_images = np.array(train_images)
train_images.shape

In [ ]:
train_images = train_images.reshape(train_images.shape[0], 400, 320, 1).astype('float32')
train_images = (train_images - 127.5) / 127.5  # Normalize the images to [-1, 1]

In [ ]:
BUFFER_SIZE = 171
BATCH_SIZE = 20

In [ ]:
# Batch and shuffle the data
train_dataset = tf.data.Dataset.from_tensor_slices(train_images).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)


In [ ]:
def make_generator_model():
    model = tf.keras.Sequential([
        layers.Dense(25*20*256, use_bias=False, input_shape=(256,)),
        layers.BatchNormalization(),
        layers.LeakyReLU(),
        layers.Reshape((25, 20, 256)),

        layers.Conv2DTranspose(128, (5, 5), strides=(2, 2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(128, (5, 5), strides=(2, 2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(32, (5, 5), strides=(2, 2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(1, (5, 5), strides=(1, 1), padding='same', use_bias=False, activation='tanh')
    ])

    print("Final output shape:", model.output_shape)
    assert model.output_shape == (None, 400, 320, 1)

    return model


#def make_generator_model():
 #   model = tf.keras.Sequential()
    # Start with a dense layer that reshapes to a smaller starting dimension than before
  #  model.add(layers.Dense(25*20*512, use_bias=False, input_shape=(100,)))  # Adjust the size to match an upscale path to 400x320
    #model.add(layers.BatchNormalization())
   # model.add(layers.LeakyReLU())

    #model.add(layers.Reshape((25, 20, 512)))  # Start with a smaller base to upscale from
    #assert model.output_shape == (None, 25, 20, 512)

    # First upscale to 50x40
    #model.add(layers.Conv2DTranspose(256, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    #model.add(layers.BatchNormalization())
    #model.add(layers.LeakyReLU())

    # Second upscale to 100x80
    #model.add(layers.Conv2DTranspose(128, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    #model.add(layers.BatchNormalization())
    #model.add(layers.LeakyReLU())

    # Third upscale to 200x160
    #model.add(layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    #model.add(layers.BatchNormalization())
    #model.add(layers.LeakyReLU())

    # Fourth upscale to 400x320
    #model.add(layers.Conv2DTranspose(3, (5, 5), strides=(2, 2), padding='same', use_bias=False, activation='tanh'))
    #assert model.output_shape == (None, 400, 320, 3)

    #return model
  

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

def make_generator_model():
    model = tf.keras.Sequential([
        # Input layer: Starting from a higher-dimensional noise vector
        layers.Dense(25*20*512, use_bias=False, input_shape=(256,)),  # Increased input dimension
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        # Reshape the output to start the convolutional process
        layers.Reshape((25, 20, 512)),

        # First Transpose Convolution: Upscale to a larger feature map
        layers.Conv2DTranspose(256, (5, 5), strides=(2, 2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        # Second Transpose Convolution: Continue upscaling
        layers.Conv2DTranspose(128, (5, 5), strides=(2, 2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        # Third Transpose Convolution: Increase spatial dimensions
        layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        # Fourth Transpose Convolution: Approach final image size
        layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        # Final Convolution: Adjust to output depth and apply activation
        layers.Conv2DTranspose(1, (5, 5), strides=(1, 1), padding='same', use_bias=False, activation='tanh'),  # Adjust channels as necessary
    ])

    return model

# Create the model
#generator = make_enhanced_generator_model()
#generator.summary()  # Print the model summary to verify the architecture


In [ ]:
generator = make_generator_model()
generator.summary()  # Print the model summary to verify the architecture

noise = tf.random.normal([1, 256])
generated_image = generator(noise, training=False)

plt.imshow(generated_image[0, :, :, 0], cmap='gray')

In [ ]:
#def make_discriminator_model():
 #   model = tf.keras.Sequential()
  #  model.add(layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same',
                                     #input_shape=[400, 320, 1]))
   # model.add(layers.LeakyReLU())
    #model.add(layers.Dropout(0.3))

 #   model.add(layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'))
 #   model.add(layers.LeakyReLU())
 #   model.add(layers.Dropout(0.3))

 #   model.add(layers.Flatten())
  #  model.add(layers.Dense(1))

   # return model




In [ ]:
def make_discriminator_model():
    model = tf.keras.Sequential([
        layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same', input_shape=[400, 320, 1]),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Conv2D(256, (5, 5), strides=(2, 2), padding='same'),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Flatten(),
        layers.Dense(1, activation='sigmoid')  # Use sigmoid for probabilistic output
    ])
    return model


In [ ]:
discriminator = make_discriminator_model()
decision = discriminator(generated_image)
print (decision)

In [ ]:
# This method returns a helper function to compute cross entropy loss
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

In [ ]:
def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    total_loss = real_loss + fake_loss
    return total_loss

In [ ]:
def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

In [ ]:
generator_optimizer = tf.keras.optimizers.Adam(1e-4)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4)

In [ ]:
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(generator_optimizer=generator_optimizer,
                                 discriminator_optimizer=discriminator_optimizer,
                                 generator=generator,
                                 discriminator=discriminator)

In [ ]:
EPOCHS = 2500
noise_dim = 256
num_examples_to_generate = 3

# You will reuse this seed overtime (so it's easier)
# to visualize progress in the animated GIF)
seed = tf.random.normal([num_examples_to_generate, noise_dim])

In [ ]:
# Notice the use of `tf.function`
# This annotation causes the function to be "compiled".
@tf.function
def train_step(images):
    noise = tf.random.normal([BATCH_SIZE, noise_dim])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
      generated_images = generator(noise, training=True)

      real_output = discriminator(images, training=True)
      fake_output = discriminator(generated_images, training=True)

      gen_loss = generator_loss(fake_output)
      disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

In [ ]:
def train(dataset, epochs):
  for epoch in range(epochs):
    start = time.time()

    for image_batch in dataset:
      train_step(image_batch)

    # Produce images for the GIF as you go
    display.clear_output(wait=True)
    generate_and_save_images(generator,
                             epoch + 1,
                             seed)

    # Save the model every 15 epochs
    if (epoch + 1) % 15 == 0:
      checkpoint.save(file_prefix = checkpoint_prefix)

    print ('Time for epoch {} is {} sec'.format(epoch + 1, time.time()-start))

  # Generate after the final epoch
  display.clear_output(wait=True)
  generate_and_save_images(generator,
                           epochs,
                           seed)

In [ ]:
import matplotlib.pyplot as plt

def generate_and_save_images(model, epoch, test_input):
    # Ensure model is in inference mode
    predictions = model(test_input, training=False)

    # Set up the figure with a more practical size
    # Here we use 15 inches by 5 inches, each image will have 5 inches width
    fig = plt.figure(figsize=(15, 5))

    # Assuming `predictions.shape[0]` is 3
    for i in range(predictions.shape[0]):
        # Create a subplot with 1 row and 3 columns
        plt.subplot(1, 3, i + 1)
        # Normalize image values to [0, 255] and display as grayscale
        plt.imshow(predictions[i, :, :, 0] * 127.5 + 127.5, cmap='gray')
        plt.axis('off')  # Hide axes

    # Save the figure
    plt.savefig('image_at_epoch_{:04d}.png'.format(epoch))
    plt.show()


In [ ]:
train(train_dataset, EPOCHS)

In [ ]:
# Display a single image using the epoch number
def display_image(epoch_no):
  return PIL.Image.open('image_at_epoch_{:04d}.png'.format(epoch_no))

In [ ]:
display_image(EPOCHS)

In [ ]:
anim_file = 'dcgan.gif'

with imageio.get_writer(anim_file, mode='I') as writer:
  filenames = glob.glob('image*.png')
  filenames = sorted(filenames)
  for filename in filenames:
    image = imageio.v2.imread(filename)
    writer.append_data(image)
  image = imageio.v2.imread(filename)
  writer.append_data(image)

In [ ]:
# To save manually
#checkpoint.save(file_prefix=checkpoint_prefix)



In [ ]:
# To save automatically during training
#manager = tf.train.CheckpointManager(checkpoint, checkpoint_dir, max_to_keep=3)
#manager.save()

In [ ]:
import numpy as np
from PIL import Image
import tensorflow as tf

def preprocess_image(image_path):
    # Load image
    img = Image.open(image_path).convert('L')  # Convert to grayscale if your model expects that
    img = img.resize((320, 400))  # Resize to the size expected by the model
    img = np.array(img)
    img = (img - 127.5) / 127.5  # Normalize the image
    img = np.expand_dims(img, axis=0)  # Add batch dimension
    img = np.expand_dims(img, axis=-1)  # Add channel dimension if needed
    return img




#image_path =  "/path/to/private-data/Croped_Pictures/train/Left/Inlet (3 marking)/example.bmp"# inlet 3
image_path = "/path/to/private-data/Croped_Pictures/train/Left/Inlet (5 marking)/example.bmp"    # inlet 5
#image_path = "/path/to/private-data/Croped_Pictures/train/Left/Outlet/example.bmp"   # Outlet


processed_image = preprocess_image(image_path)

In [ ]:
def evaluate_image(model, image):
    prediction = model(image, training=False)  # Set training to False
    return prediction

discriminator_result = evaluate_image(discriminator, processed_image)
print("Discriminator output:", discriminator_result.numpy())


In [ ]:
import glob, os
 
in3 = []
in5 = []
outlet = []


path = "/path/to/private-data/Croped_Pictures/train/Left/*"
for directory_path in glob.glob(path):
    label=directory_path.split("\\")[-1]
    for image_path in glob.glob(os.path.join(directory_path,"*.bmp")):
        img = preprocess_image(image_path)
        prediction = evaluate_image(discriminator, img)
        prediction = prediction.numpy()
        
        if label == "Inlet (3 marking)":
            in3.append(prediction)
            
        elif label == "Inlet (5 marking)":
            in5.append(prediction)
            
        elif label == "Outlet":
            outlet.append(prediction)

print('Done')

In [ ]:
val_in3 = []
val_in5 = []
val_outlet= []
val_others = []
val_wrp = []

path = "/path/to/private-data/Croped_Pictures/validation/Left/*"
for directory_path in glob.glob(path):
    label=directory_path.split("\\")[-1]
    for image_path in glob.glob(os.path.join(directory_path,"*.bmp")):
        img = preprocess_image(image_path)
        prediction = evaluate_image(discriminator, img)
        prediction = prediction.numpy()

        if label == "Inlet (3 marking)":
            val_in3.append(prediction) 
            
        elif label == "Inlet (5 marking)":
            val_in5.append(prediction)

        elif label == "Outlet":
            val_outlet.append(prediction)

        elif label == "Others":
            val_others.append(prediction)
        
        elif label == "wrong placement":
            val_wrp.append(prediction)
            
print('Done')

In [ ]:
in3 = np.squeeze(in3)
in5 = np.squeeze(in5)
outlet = np.squeeze(outlet)

val_in3 = np.squeeze(val_in3)
val_in5 = np.squeeze(val_in5)
val_outlet = np.squeeze(val_outlet)
val_wrp = np.squeeze(val_wrp)
val_others = np.squeeze(val_others)

In [ ]:
import seaborn as sns
plt.figure(figsize=(8, 6))

# Create the box plot
sns.boxplot(data=[in3, in5, outlet, val_outlet, val_others])

# Setting labels
plt.xlabel('Class')
plt.ylabel('MSE Scores')
plt.title('Box Plot of MSE Scores for Different Classes')
plt.xticks([0, 1, 2,3 ,4], ['Inlet type-1', 'Inlet Type-2', 'Outlet', 'Outlet (Test)','Unmarked'])  # Label x-ticks with class names

# Show the plot
plt.show()

In [ ]:
import seaborn as sns
plt.figure(figsize=(8, 6))

# Create the box plot
sns.boxplot(data=[val_in3, val_in5, val_in5,  val_outlet,val_others, val_wrp])

# Setting labels
plt.xlabel('Class')
plt.ylabel('MSE Scores')
plt.title('Box Plot of MSE Scores for Different Classes')
plt.xticks([0, 1, 2,3 ,4, 5], ['Inlet type-1', 'Inlet Type-2','Inlet Type-2 (Test)' ,'Outlet','Unmarked', 'Wrong placement'])  # Label x-ticks with class names

# Show the plot
plt.show()

In [ ]:
print(val_in5.min())

print(val_wrp.max())
print(val_others.max())
print(val_in3.max())


In [ ]:
import plotly.express as px
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import pandas as pd

# Assuming 'data' is your high-dimensional dataset
pca_50 = PCA(n_components=25)
data_50 = pca_50.fit_transform(data)

# Apply t-SNE for 3 dimensions
tsne = TSNE(n_components=3, verbose=1, perplexity=40, n_iter=1000)
tsne_results = tsne.fit_transform(data_50)

# Create a DataFrame for easier plotting
df_tsne = pd.DataFrame(tsne_results, columns=['Dimension 1', 'Dimension 2', 'Dimension 3'])
df_tsne['Label'] = label_data  # your label data

# Map numeric labels to descriptive labels
label_map = {
    0: "Inlet Type-2",
    1: "Inlet Type-1",
    2: "Outlet",
    3: "Inlet Type-2 (Test)",
    4: "Unmarked"
}
df_tsne['Label'] = df_tsne['Label'].map(label_map)

# Plot using Plotly Express
fig = px.scatter_3d(
    df_tsne,
    x='Dimension 1',
    y='Dimension 2',
    z='Dimension 3',
    color='Label',  # Color by the 'Label' column
    color_discrete_map={  # Optional: custom color map
        "Inlet Type-2": 'blue',
        "Inlet Type-1": 'orange',
        "Outlet": 'green',
        "Inlet Type-2 (Test)": 'red',
        "Unmarked": 'purple'
    },
    title='3D t-SNE Visualization'
)

fig.update_traces(marker=dict(size=3))  # Adjust marker size here
fig.show()
